# GP00 Task Audit

Audit the task inventory and construct the provisional development and evaluation split.

This source copy comes from **dataset-verification.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

**Clear-text source release:** No embedded ZIP, Base64 or hex payloads are required to read the notebook source.


### Preserved execution evidence — source archive

- **Executed source:** `Paper/GP00/dataset-verification.ipynb` in the author-provided `Paper (2)(2).zip` (10 November 2026).
- **Saved results:** 8 human-readable Jupyter output records restored from matching executed cells.
- **Source fidelity:** 7 identical code cells; 0 proven value-equivalent readable source strings; 0 historical configuration cell(s) restored to align with archived execution.
- **Excluded display artifacts:** 0 active, encoded, or binary browser-download outputs were omitted; the source archive retains them. These are *not* replacements for the sealed experiment ZIPs.
- **Reproduction distinction:** Saved historical output does **not** mean this notebook has been rerun in the current environment. The original experiment may require Kaggle mounts, GPUs, offline wheels and authorised archives. The additional execution guide above details the prerequisites.


## How to execute or reproduce this implementation stage

**Execution mode:** CPU.  
**Prerequisites:** Kaggle competition dataset mounted with the same published tasks.jsonl snapshot.

**Steps:** Run all cells from top to bottom after attaching the competition input; check the 129-task inventory and deterministic split. A changed dataset snapshot may change results.

**Reproduction boundary:** Conditional: public dataset required.

> **Historical evidence:** This release retains implementation source but removes historical cell outputs. Results from stochastic GPU runs are not guaranteed to repeat byte-for-byte. Use the sealed evidence records and the E17 reviewer companion to reconstruct accepted historical outcomes.


# GP00 — Gemma 4 Paper: Dataset and Harness Audit

**Version 1.0 | 5 November 2026 | Clifford Imaguezegie**

This is our first execution step. Use **CPU / no accelerator**. No API key, model download, inference, package installation or paid service is needed.

The notebook audits the published developer-agent dataset, records source fingerprints, and proposes task candidates without reading reference-fix or evaluator-test contents as research features. It never exports those contents. It does not extract snapshots, execute tests or qualify tasks for main evaluation.

## Before running

1. Sign in to Kaggle. If needed, accept the rules for the **companion coding competition**, which supplies the task dataset.
2. Import this `.ipynb` into a new Kaggle Notebook.
3. Attach **Google — The Gemma 4 Developer Agent Competition** data through Add Input / Add Data. The paper-track page may not have its own task download.
4. Choose CPU/no accelerator. Internet can remain off once the data is attached.
5. Run Cells 1–7 in order. If more than one `tasks.jsonl` is found, set `TASKS_PATH` in Cell 1 to the **published development** file and rerun from Cell 3. Do not choose a private/hidden evaluation file.

The expected published count is 129 records; a mismatch is recorded for review rather than silently treated as the same release. Exact task IDs and environment commands are not predetermined.


## Cell 1 — Configuration
Leave `TASKS_PATH = None` for automatic discovery when exactly one task file is attached.


In [1]:
from pathlib import Path
import json

COMPETITION_ROOT = Path(
    '/kaggle/input/competitions/gemma-4-developer-agent'
)

task_files = sorted(
    p for p in COMPETITION_ROOT.rglob('tasks.jsonl')
    if p.is_file()
)

if len(task_files) != 1:
    raise RuntimeError(
        f"Expected one tasks.jsonl; found {len(task_files)}. "
        f"Files: {[str(p) for p in task_files]}"
    )

TASKS_PATH = task_files[0]

OUTPUT_BASE = (
    Path('/kaggle/working')
    if Path('/kaggle/working').exists()
    else Path.cwd()
) / 'Gemma4_Paper'

SEED = 20261105
DEVELOPMENT_CANDIDATES = 6
EVALUATION_CANDIDATES = 36
EXPECTED_PUBLISHED_ROWS = 129

print(json.dumps({
    'cell': 1,
    'mode': 'CPU_METADATA_AUDIT',
    'tasks_override': str(TASKS_PATH),
    'output_base': str(OUTPUT_BASE),
    'seed': SEED,
    'model_calls': 0
}, indent=2))

{
  "cell": 1,
  "mode": "CPU_METADATA_AUDIT",
  "tasks_override": "/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl",
  "output_base": "/kaggle/working/Gemma4_Paper",
  "seed": 20261105,
  "model_calls": 0
}


**Observation — complete after execution:** Confirm the CPU audit mode and output location. No data has been read and no model has been called.

This notebook is delivered without competition execution outputs.


## Cell 2 — Load the audit functions
The following code uses only Python's standard library. It defines the audit functions without executing repository code.


In [2]:
"""GP00 v1.0: metadata audit only. Standard library; no model calls or test execution.

Competition task answers are never exported. Candidate splits are provisional,
and no task is execution-qualified by this audit.
"""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import re
import sys
import uuid
import zipfile

VERSION = "GP00_v1.0"
DEFAULT_SEED = 20261105
SAFE_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,179}\Z")
SAFE_REPO = re.compile(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+\Z")
HEX_SHA = re.compile(r"[a-fA-F0-9]{40}\Z")
AGENT_ALLOWLIST = ("instance_id", "repo", "base_commit", "problem_statement")
AUDIT_FIELDS = (
    "row_number", "instance_id", "repo", "base_commit", "split_group",
    "problem_chars", "reference_patch_present", "verification_patch_present",
    "test_command_status", "snapshot_path", "snapshot_bytes", "graph_path",
    "graph_nodes", "graph_edges", "graph_error", "embedding_path",
    "metadata_candidate", "hold_reasons", "execution_qualified",
)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def stable_rank(seed: int, value: str) -> str:
    return hashlib.sha256(f"{seed}:{value}".encode()).hexdigest()


def discover_tasks(explicit=None, roots=None) -> Path:
    if explicit:
        p = Path(explicit).expanduser().resolve()
        p = p / "tasks.jsonl" if p.is_dir() else p
        if not p.is_file() or p.name != "tasks.jsonl":
            raise FileNotFoundError("Set TASKS_PATH to the published tasks.jsonl file or its directory.")
        return p
    roots = roots if roots is not None else [Path("/kaggle/input"), Path("/content")]
    found = set()
    for root in map(Path, roots):
        if root.is_dir():
            found.update(p.resolve() for p in root.rglob("tasks.jsonl") if p.is_file())
    if len(found) != 1:
        visible = "\n".join(str(p) for p in sorted(found)) or "No tasks.jsonl was found."
        raise RuntimeError(
            "Select the published development dataset explicitly. Automatic selection needs exactly one file.\n"
            + visible + "\nSet TASKS_PATH in Cell 1 and rerun from Cell 3."
        )
    return next(iter(found))


def load_metadata(tasks_path: Path):
    """Only whitelisted fields/presence flags survive each JSONL iteration."""
    records = []
    seen_ids = set()
    keys = Counter()
    with tasks_path.open(encoding="utf-8-sig") as stream:
        for line_no, line in enumerate(stream, 1):
            if not line.strip():
                continue
            try:
                raw = json.loads(line)
            except (json.JSONDecodeError, ValueError) as exc:
                raise ValueError(f"Invalid JSON at line {line_no}; raw task contents suppressed.") from exc
            if not isinstance(raw, dict):
                raise ValueError(f"Line {line_no} is not a JSON object; contents suppressed.")
            keys.update(raw.keys())
            item_id = raw.get("instance_id")
            if isinstance(item_id, str) and item_id:
                if item_id in seen_ids:
                    raise ValueError(f"Duplicate instance_id at line {line_no}: {item_id}")
                seen_ids.add(item_id)
            issues = []
            for key, pattern in (("instance_id", SAFE_ID), ("repo", SAFE_REPO), ("base_commit", HEX_SHA)):
                v = raw.get(key)
                if not isinstance(v, str) or not pattern.fullmatch(v):
                    issues.append("invalid_" + key)
            statement = raw.get("problem_statement")
            if not isinstance(statement, str) or not statement.strip():
                issues.append("missing_problem_statement")
            command = raw.get("test_command")
            if "test_command" not in raw:
                command_status = "ABSENT_harness_resolution_pending"
            elif isinstance(command, str) and command.strip():
                command_status = "PRESENT_not_executed"
            else:
                command_status = "EMPTY_OR_INVALID_requires_review"
                issues.append("provided_test_command_needs_review")
            # Inspect presence only; do not retain answer text or hidden test text.
            reference_present = isinstance(raw.get("patch"), str) and bool(raw["patch"].strip())
            verification_present = isinstance(raw.get("test_patch"), str) and bool(raw["test_patch"].strip())
            if not verification_present:
                issues.append("verification_material_missing")
            repo = raw.get("repo") if isinstance(raw.get("repo"), str) else ""
            commit = raw.get("base_commit") if isinstance(raw.get("base_commit"), str) else ""
            record = {
                "row_number": line_no,
                "instance_id": item_id if isinstance(item_id, str) else "",
                "repo": repo, "base_commit": commit,
                "split_group": f"{repo}@{commit}",
                "problem_chars": len(statement) if isinstance(statement, str) else 0,
                "reference_patch_present": reference_present,
                "verification_patch_present": verification_present,
                "test_command_status": command_status,
                "hold_reasons": issues,
                "execution_qualified": False,
            }
            records.append(record)
            del raw, line, statement, command
    if not records:
        raise ValueError("tasks.jsonl contains no task objects.")
    return records, dict(sorted(keys.items()))


def asset_roots(tasks_path: Path):
    # Documented layouts: ROOT/tasks.jsonl and ROOT/published/tasks.jsonl.
    base = tasks_path.parent.resolve()
    roots = [base]
    if base.name.lower() in {"published", "train", "training"}:
        roots.append(base.parent)
    return roots


def resolve_asset(roots, relative_names):
    for root in roots:
        for name in relative_names:
            p = (root / name).resolve()
            if p.is_relative_to(root.resolve()) and p.is_file() and p.stat().st_size > 0:
                return p
    return None


def inspect_graph(path: Path):
    result = {"graph_nodes": 0, "graph_edges": 0, "graph_error": ""}
    try:
        if path.stat().st_size > 128 * 1024 * 1024:
            raise ValueError("graph_exceeds_128MiB_audit_limit")
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            raise ValueError("graph_not_object")
        nodes = data.get("nodes")
        edges = data.get("edges", data.get("links"))
        if not isinstance(nodes, list) or not nodes:
            raise ValueError("graph_has_no_nodes")
        if not isinstance(edges, list):
            raise ValueError("graph_has_no_edge_list")
        ids = [n.get("id") if isinstance(n, dict) else None for n in nodes]
        if any(not isinstance(n, str) or not n for n in ids) or len(set(ids)) != len(ids):
            raise ValueError("graph_node_ids_invalid_or_duplicated")
        id_set = set(ids)
        if any(not isinstance(e, dict) or e.get("source") not in id_set or e.get("target") not in id_set for e in edges):
            raise ValueError("graph_edge_endpoint_invalid")
        result.update(graph_nodes=len(nodes), graph_edges=len(edges))
    except (OSError, ValueError, TypeError, UnicodeError) as exc:
        # No source snippets or answer text in diagnostics.
        result["graph_error"] = str(exc) if isinstance(exc, ValueError) and str(exc).startswith("graph_") else type(exc).__name__
    return result


def audit_assets(records, tasks_path: Path):
    roots = asset_roots(tasks_path)
    manifest = []
    cached_graphs = {}
    for row in records:
        item_id = row["instance_id"]
        if not SAFE_ID.fullmatch(item_id):
            row.update(snapshot_path="", snapshot_bytes=0, graph_path="", graph_nodes=0,
                       graph_edges=0, graph_error="unsafe_id_not_resolved", embedding_path="")
        else:
            short = row["repo"].rsplit("/", 1)[-1]
            alias = f"{short}_{row['base_commit']}"
            snapshot = resolve_asset(roots, [f"snapshots/{item_id}.tgz", f"snapshots/{item_id}.tar.gz", f"snapshots/{item_id}.tar"])
            graph = resolve_asset(roots, [f"graphs/{item_id}.json", f"graphs/{alias}.json"])
            embedding = resolve_asset(roots, [f"embeddings/{item_id}.npz", f"embeddings/{alias}.npz"])
            info = {"graph_nodes": 0, "graph_edges": 0, "graph_error": "missing_graph"}
            if graph:
                if graph not in cached_graphs:
                    cached_graphs[graph] = (inspect_graph(graph), sha256_file(graph))
                info = dict(cached_graphs[graph][0])
            row.update(snapshot_path=str(snapshot or ""), snapshot_bytes=snapshot.stat().st_size if snapshot else 0,
                       graph_path=str(graph or ""), embedding_path=str(embedding or ""), **info)
            if not snapshot:
                row["hold_reasons"].append("snapshot_missing_or_empty")
            if info["graph_error"]:
                row["hold_reasons"].append("graph_missing_or_invalid")
            for kind, p in (("snapshot", snapshot), ("graph", graph), ("embedding", embedding)):
                if p:
                    manifest.append({"instance_id": item_id, "kind": kind, "path": str(p),
                                     "bytes": p.stat().st_size,
                                     "sha256": cached_graphs[p][1] if kind == "graph" else None,
                                     "validation": "JSON_structure_only" if kind == "graph" else "existence_and_size_only"})
        row["metadata_candidate"] = not row["hold_reasons"]
    return records, manifest


def runtime_inventory():
    versions = {}
    for package in ("swegemma", "adk-submission", "adk-eval-core", "google-adk", "vllm", "torch"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not_installed"
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "packages": versions, "environment_variables_read": [],
            "gpu_probed": False, "model_loaded": False}


def inspect_harness(tasks_path: Path):
    roots = asset_roots(tasks_path)
    readme = resolve_asset(roots, ["HARNESS_README.md"])
    info = {"readme_path": str(readme or ""), "readme_sha256": None,
            "wheel_fingerprints": [], "execution_api_verified": False}
    readme_text = ""
    if readme:
        if readme.stat().st_size <= 2 * 1024 * 1024:
            readme_text = readme.read_text(encoding="utf-8")
            info["readme_sha256"] = sha256_file(readme)
        else:
            info["readme_warning"] = "readme_exceeds_2MiB_audit_limit"
    seen = set()
    for root in roots:
        for wheel in sorted((root / "wheels").glob("*.whl")):
            if wheel.name.lower().startswith(("swegemma-", "adk_submission-", "adk_eval_core-")) and wheel.resolve() not in seen:
                seen.add(wheel.resolve())
                info["wheel_fingerprints"].append({"name": wheel.name, "bytes": wheel.stat().st_size, "sha256": sha256_file(wheel)})
    return info, readme_text


def candidate_split(records, seed=DEFAULT_SEED, dev_target=6, eval_target=36):
    if not isinstance(seed, int) or isinstance(seed, bool) or min(dev_target, eval_target) < 1:
        raise ValueError("Use an integer seed and positive candidate targets.")
    grouped = defaultdict(list)
    for row in records:
        if row["metadata_candidate"]:
            grouped[row["split_group"]].append(row)
    representatives = {g: min(rows, key=lambda r: stable_rank(seed, r["instance_id"])) for g, rows in grouped.items()}
    by_repo = defaultdict(list)
    for group, representative in representatives.items():
        by_repo[representative["repo"]].append(group)
    queues = {repo: sorted(groups, key=lambda g: stable_rank(seed, g)) for repo, groups in sorted(by_repo.items())}
    order = []
    while any(queues.values()):
        for repo in sorted(queues):
            if queues[repo]:
                order.append(queues[repo].pop(0))
    group_role = {g: "reserve" for g in order}
    for group in order[:dev_target]:
        group_role[group] = "development_candidate"
    for group in order[dev_target:dev_target + eval_target]:
        group_role[group] = "evaluation_candidate"
    result = []
    for row in records:
        group = row["split_group"]
        role = group_role.get(group, "metadata_hold")
        selected = role in {"development_candidate", "evaluation_candidate"} and representatives[group]["instance_id"] == row["instance_id"]
        result.append({"instance_id": row["instance_id"], "repo": row["repo"], "base_commit": row["base_commit"],
                       "split_group": group, "group_role": role, "candidate_selected": selected,
                       "execution_qualified": False, "split_frozen": False, "seed": seed})
    return result


def _write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def _write_csv(path, rows, fields):
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            out = {key: row.get(key, "") for key in fields}
            if isinstance(out.get("hold_reasons"), list):
                out["hold_reasons"] = ";".join(out["hold_reasons"])
            writer.writerow(out)


def export_audit(tasks_path, records, field_counts, assets, harness, readme_text, split,
                 output_base, seed=DEFAULT_SEED, expected_count=129):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    folder = Path(output_base).expanduser().resolve() / f"GP00_{stamp}_{uuid.uuid4().hex[:8]}"
    folder.mkdir(parents=True, exist_ok=False)
    selected = [s for s in split if s["candidate_selected"]]
    warnings = []
    if len(records) != expected_count:
        warnings.append(f"Observed {len(records)} rows; published expectation was {expected_count}. Review dataset version before freezing.")
    if not harness["readme_sha256"]:
        warnings.append("HARNESS_README.md was not captured; harness review is required.")
    if sum(s["group_role"] == "development_candidate" for s in selected) < 6:
        warnings.append("Fewer than six development candidates; do not silently relax the protocol.")
    if sum(s["group_role"] == "evaluation_candidate" for s in selected) < 36:
        warnings.append("Fewer than 36 evaluation candidates; confirm eligibility and feasible target before freezing.")
    repos = sorted(set(s["repo"] for s in selected))
    if len(repos) < 3:
        warnings.append("Candidates span fewer than three repositories.")
    summary = {
        "version": VERSION, "generated_utc": stamp, "status": "METADATA_AUDIT_ONLY",
        "tasks_file": str(tasks_path), "tasks_sha256": sha256_file(tasks_path),
        "rows_observed": len(records), "expected_rows_from_documentation": expected_count,
        "repos": dict(sorted(Counter(r["repo"] for r in records).items())),
        "field_presence_counts": field_counts,
        "metadata_candidates": sum(r["metadata_candidate"] for r in records),
        "metadata_holds": sum(not r["metadata_candidate"] for r in records),
        "test_command_status_counts": dict(Counter(r["test_command_status"] for r in records)),
        "candidate_roles": dict(Counter(s["group_role"] for s in selected)),
        "unique_candidate_groups": len(set(s["split_group"] for s in selected)),
        "seed": seed, "split_frozen": False, "execution_qualified_tasks": 0,
        "reference_patches_exported": False, "verification_patches_exported": False,
        "test_commands_executed": 0, "model_calls": 0, "warnings": warnings,
    }
    _write_json(folder / "00_summary.json", summary)
    _write_csv(folder / "01_task_audit.csv", records, AUDIT_FIELDS)
    _write_csv(folder / "02_candidate_split.csv", split, tuple(split[0]))
    _write_json(folder / "03_asset_manifest.json", {"tasks_sha256": summary["tasks_sha256"], "assets": assets, "harness": harness})
    (folder / "04_harness_readme.md").write_text(readme_text or "HARNESS_README.md NOT FOUND.\n", encoding="utf-8")
    _write_json(folder / "05_runtime_inventory.json", runtime_inventory())
    _write_json(folder / "07_agent_input_allowlist.json", {"allowed_fields": AGENT_ALLOWLIST,
                "excluded": ["patch", "test_patch", "test_command", "hints_text"],
                "note": "Schema only. No task prompts or solutions are exported by GP00. Public repository tests may be used in later agent sandboxes."})
    notes = (
        f"# GP00 observations\n\nRun: {stamp}; script: {VERSION}.\n\n"
        f"Observed **{len(records)}** task records; **{summary['metadata_candidates']}** metadata candidates.\n"
        f"Metadata holds: **{summary['metadata_holds']}**. Candidate assignments: {summary['candidate_roles']}.\n\n"
        "## Meaning of this output\n\n"
        "This is a metadata audit, not a repair experiment. No repository was extracted, installed, edited, or tested; no model was loaded or called. "
        "Snapshot and embedding checks establish existence and size only. Graph checks establish JSON structure only. "
        "No task has been execution-qualified and no split is frozen. A missing test_command may be resolved by the official harness; it does not prove a task is unscorable.\n\n"
        "## Next review\n\n"
        "Review the captured harness, candidate IDs, same-commit groups, and holds. Then qualify evaluator commands in isolated sandboxes, "
        "checking buggy-baseline failure and reference-fix success using evaluator-only material. Freeze the protocol and split before main model runs.\n\n"
        "## Warnings\n\n" + ("\n".join("- " + w for w in warnings) if warnings else "None at metadata stage.") + "\n"
    )
    (folder / "06_observations.md").write_text(notes, encoding="utf-8")
    package = folder / "GP00_results.zip"
    with zipfile.ZipFile(package, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(folder.iterdir()):
            if p.is_file() and p != package:
                archive.write(p, arcname=p.name)
    return folder, summary


def run_audit(tasks=None, output=None, seed=DEFAULT_SEED, dev_target=6, eval_target=36):
    tasks_path = discover_tasks(tasks)
    records, field_counts = load_metadata(tasks_path)
    records, assets = audit_assets(records, tasks_path)
    harness, text = inspect_harness(tasks_path)
    split = candidate_split(records, seed, dev_target, eval_target)
    base = output or (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()) / "Gemma4_Paper"
    return export_audit(tasks_path, records, field_counts, assets, harness, text, split, base, seed)


print(json.dumps({'cell': 2, 'audit_version': VERSION, 'functions_loaded': True, 'model_calls': 0}))


{"cell": 2, "audit_version": "GP00_v1.0", "functions_loaded": true, "model_calls": 0}


**Observation — complete after execution:** The functions-loaded message should appear. A Python error here is a notebook issue, not a competition result.

This notebook is delivered without competition execution outputs.


## Cell 3 — Locate the published task file
Automatic discovery refuses ambiguous inputs rather than selecting a release silently.


In [3]:
TASK_FILE = discover_tasks(TASKS_PATH)
print(json.dumps({'cell': 3, 'tasks_file': str(TASK_FILE), 'bytes': TASK_FILE.stat().st_size,
                  'sha256': sha256_file(TASK_FILE)}, indent=2))


{
  "cell": 3,
  "tasks_file": "/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl",
  "bytes": 1984455,
  "sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6"
}


**Observation — complete after execution:** Confirm this is the published development dataset. If no file is found, attach the coding-competition data. If several files are listed, set TASKS_PATH explicitly; do not continue with an unknown release.

This notebook is delivered without competition execution outputs.


## Cell 4 — Audit metadata and asset structure
Reference-patch and verification-patch **presence** is checked; their text is not retained or exported. No task code is executed.


In [4]:
RECORDS, FIELD_COUNTS = load_metadata(TASK_FILE)
RECORDS, ASSET_MANIFEST = audit_assets(RECORDS, TASK_FILE)
print(json.dumps({'cell': 4, 'rows_observed': len(RECORDS), 'expected_rows': EXPECTED_PUBLISHED_ROWS,
                  'repositories': dict(sorted(Counter(r['repo'] for r in RECORDS).items())),
                  'metadata_candidates': sum(r['metadata_candidate'] for r in RECORDS),
                  'metadata_holds': sum(not r['metadata_candidate'] for r in RECORDS),
                  'command_status_counts': dict(Counter(r['test_command_status'] for r in RECORDS)),
                  'execution_qualified_tasks': 0}, indent=2))


{
  "cell": 4,
  "rows_observed": 129,
  "expected_rows": 129,
  "repositories": {
    "Textualize/rich": 48,
    "encode/httpx": 1,
    "fastapi/fastapi": 67,
    "psf/requests": 13
  },
  "metadata_candidates": 129,
  "metadata_holds": 0,
  "command_status_counts": {
    "ABSENT_harness_resolution_pending": 129
  },
  "execution_qualified_tasks": 0
}


**Observation — complete after execution:** Record the actual count, repository coverage and holds. Metadata candidates have not passed execution checks. An absent test command can be resolved by the harness; it does not establish that a task is unscorable.

This notebook is delivered without competition execution outputs.


## Cell 5 — Capture harness fingerprints and runtime inventory
Copies the public harness README into the eventual audit package and hashes the relevant harness wheels if present. It does not install or run them.


In [5]:
HARNESS_INFO, HARNESS_TEXT = inspect_harness(TASK_FILE)
RUNTIME_INFO = runtime_inventory()
print(json.dumps({'cell': 5, 'harness': HARNESS_INFO, 'runtime': RUNTIME_INFO}, indent=2))


{
  "cell": 5,
  "harness": {
    "readme_path": "/kaggle/input/competitions/gemma-4-developer-agent/HARNESS_README.md",
    "readme_sha256": "445000219b803942c469391ec8b69791a31e9cec2f934738c767e34b5ca0aa1f",
    "wheel_fingerprints": [],
    "execution_api_verified": false
  },
  "runtime": {
    "python": "3.13.18",
    "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
    "packages": {
      "swegemma": "not_installed",
      "adk-submission": "not_installed",
      "adk-eval-core": "not_installed",
      "google-adk": "2.7.1",
      "vllm": "not_installed",
      "torch": "2.11.0+cpu"
    },
    "environment_variables_read": [],
    "gpu_probed": false,
    "model_loaded": false
  }
}


**Observation — complete after execution:** Check whether HARNESS_README.md and harness wheels were found. Installed package versions alone cannot identify repackaged wheels. Missing packages are expected in this CPU-only audit.

This notebook is delivered without competition execution outputs.


## Cell 6 — Propose a reproducible candidate split
Candidates are balanced across repositories, ranked using a fixed hash seed, and grouped by repository/base commit. Reference answers and model outcomes are not selection features.


In [6]:
CANDIDATES = candidate_split(RECORDS, SEED, DEVELOPMENT_CANDIDATES, EVALUATION_CANDIDATES)
SELECTED = [r for r in CANDIDATES if r['candidate_selected']]
DEV_GROUPS = {r['split_group'] for r in SELECTED if r['group_role'] == 'development_candidate'}
EVAL_GROUPS = {r['split_group'] for r in SELECTED if r['group_role'] == 'evaluation_candidate'}
assert DEV_GROUPS.isdisjoint(EVAL_GROUPS), 'Development/evaluation group overlap detected.'
print(json.dumps({'cell': 6, 'candidate_roles': dict(Counter(r['group_role'] for r in SELECTED)),
                  'repository_coverage': dict(Counter(r['repo'] for r in SELECTED)),
                  'group_overlap': len(DEV_GROUPS & EVAL_GROUPS), 'split_frozen': False}, indent=2))
print('Development candidates:', [r['instance_id'] for r in SELECTED if r['group_role'] == 'development_candidate'])
print('Evaluation candidates:', [r['instance_id'] for r in SELECTED if r['group_role'] == 'evaluation_candidate'])


{
  "cell": 6,
  "candidate_roles": {
    "evaluation_candidate": 36,
    "development_candidate": 6
  },
  "repository_coverage": {
    "fastapi/fastapi": 14,
    "psf/requests": 12,
    "Textualize/rich": 15,
    "encode/httpx": 1
  },
  "group_overlap": 0,
  "split_frozen": false
}
Development candidates: ['fastapi_14479', 'fastapi_5624', 'httpx_3672', 'requests_6757', 'rich_2943', 'rich_3130']
Evaluation candidates: ['fastapi_14964', 'fastapi_15023', 'requests_7505', 'requests_7502', 'requests_7433', 'requests_7427', 'requests_7315', 'requests_7328', 'requests_7309', 'requests_7205', 'rich_4070', 'rich_4077', 'fastapi_14258', 'fastapi_14794', 'fastapi_14463', 'fastapi_12942', 'fastapi_14482', 'fastapi_14458', 'fastapi_14266', 'rich_3718', 'rich_3953', 'rich_3944', 'fastapi_15763', 'fastapi_15745', 'fastapi_14303', 'requests_6644', 'requests_6589', 'requests_6592', 'rich_3772', 'rich_3535', 'rich_3506', 'rich_3470', 'rich_3278', 'rich_3043', 'rich_3067', 'rich_3006']


**Observation — complete after execution:** Record the actual proposed task IDs. The split remains provisional until the evaluator, repository setup and grouping are reviewed. Do not start model runs or manually swap tasks based on expected ease.

This notebook is delivered without competition execution outputs.


## Cell 7 — Export the audit package
A new uniquely named folder is created on each export. No old result is overwritten.


In [7]:
RESULT_DIR, SUMMARY = export_audit(TASK_FILE, RECORDS, FIELD_COUNTS, ASSET_MANIFEST,
                                  HARNESS_INFO, HARNESS_TEXT, CANDIDATES, OUTPUT_BASE,
                                  SEED, EXPECTED_PUBLISHED_ROWS)
print(json.dumps({'cell': 7, 'event': 'GP00_COMPLETE', 'output_directory': str(RESULT_DIR),
                  'results_zip': str(RESULT_DIR / 'GP00_results.zip'), 'summary': SUMMARY}, indent=2))
print((RESULT_DIR / '06_observations.md').read_text(encoding='utf-8'))
try:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULT_DIR / 'GP00_results.zip')))
except ImportError:
    print('Download GP00_results.zip from the output directory shown above.')


{
  "cell": 7,
  "event": "GP00_COMPLETE",
  "output_directory": "/kaggle/working/Gemma4_Paper/GP00_20261105T103909Z_df8aed0a",
  "results_zip": "/kaggle/working/Gemma4_Paper/GP00_20261105T103909Z_df8aed0a/GP00_results.zip",
  "summary": {
    "version": "GP00_v1.0",
    "generated_utc": "20261105T103909Z",
    "status": "METADATA_AUDIT_ONLY",
    "tasks_file": "/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl",
    "tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6",
    "rows_observed": 129,
    "expected_rows_from_documentation": 129,
    "repos": {
      "Textualize/rich": 48,
      "encode/httpx": 1,
      "fastapi/fastapi": 67,
      "psf/requests": 13
    },
    "field_presence_counts": {
      "base_commit": 129,
      "created_at": 129,
      "hints_text": 129,
      "instance_id": 129,
      "patch": 129,
      "problem_statement": 129,
      "repo": 129,
      "test_patch": 129
    },
    "metadata_candidates": 129,
    "metadata_

/kaggle/working/Gemma4_Paper/GP00_20261105T103909Z_df8aed0a/GP00_results.zip

**Observation — complete after execution:** Confirm GP00_COMPLETE, zero model calls and zero execution-qualified tasks. Preserve the printed observations and download the ZIP. If warnings appear, include them in the review; they are not instructions to bypass a check.

This notebook is delivered without competition execution outputs.


## What to return

Return **GP00_results.zip** from this notebook's output folder. It contains:

- The observed dataset count and SHA-256.
- A metadata audit for every record and the proposed candidate split.
- Asset/harness fingerprints and the public harness README, if found.
- Runtime inventory, generated observations and the later agent-input allowlist.

It excludes the original tasks file, task prompts, reference fixes, evaluator test patches, repository archives, embeddings and model weights. No software-repair result has been measured by this notebook.

**Next:** review this output, implement the evaluator against the actual captured harness, and qualify tasks before freezing the experiment. Do not run GP01/model experiments yet; those notebooks are not part of this first step.
